# Módulo 3 · Clase 4 — Bash scripting

**Curso de Inteligencia Artificial — Nivel básico**

---

## ¿Qué vamos a ver hoy?

| # | Bloque | Qué aprenderás |
|---|--------|----------------|
| 1 | Estructura de un script Bash | Shebang, comentarios, `chmod +x` |
| 2 | Variables | Declarar, usar, variables especiales |
| 3 | Argumentos | `$1`, `$2`, `$@`, `$#` |
| 4 | Condicionales | `if`, `elif`, `else`, operadores de comparación |
| 5 | Bucles | `for`, `while` |
| 6 | Funciones | Declarar y llamar funciones |
| 7 | Práctica | Scripts de automatización para proyectos IA |

> 💡 **Idea central:** Un script Bash es la forma más directa de automatizar tareas repetitivas en el servidor. En IA lo usarás para lanzar experimentos, procesar datos, hacer backups automáticos y preparar entornos.

---
## 1. Estructura de un script Bash

```bash
#!/bin/bash
# Mi primer script — comentario descriptivo

echo "Hola desde Bash"
```

**Línea 1: el shebang `#!/bin/bash`**  
Le dice al sistema qué intérprete usar. Sin él, el sistema intentará adivinar y puede fallar.

```bash
#!/bin/bash        # Bash (el más común)
#!/usr/bin/env bash # Busca bash en el PATH (más portable)
#!/bin/sh          # Shell POSIX básico (compatible en más sistemas)
#!/usr/bin/env python3  # También funciona con Python
```

**Para ejecutar un script:**
```bash
# Opción 1: hacerlo ejecutable y correrlo
chmod +x mi_script.sh
./mi_script.sh

# Opción 2: pasarlo a bash directamente
bash mi_script.sh
```

**Buenas prácticas desde el principio:**
```bash
#!/usr/bin/env bash
set -e    # salir si cualquier comando falla
set -u    # error si se usa variable no definida
set -o pipefail  # detectar fallos en pipes
```

In [ ]:
from pathlib import Path
import os, subprocess

SCRIPTS = Path.home() / "bash_scripts"
SCRIPTS.mkdir(exist_ok=True)
os.chdir(SCRIPTS)

# Escribir el primer script
Path("hola.sh").write_text("""\
#!/usr/bin/env bash
# Primer script: saludo básico

echo "Hola desde Bash"
echo "Fecha: $(date +%Y-%m-%d)"
echo "Usuario: $USER"
echo "Directorio: $(pwd)"
""", encoding="utf-8")

!chmod +x hola.sh
print("Script creado y ejecutable. Ejecutando:")
!bash hola.sh

---
## 2. Variables en Bash

```bash
# Declarar (sin espacios alrededor del =)
NOMBRE="Ana"
NUMERO=42
LISTA="a b c"

# Usar (con el prefijo $)
echo "Hola, $NOMBRE"
echo "El número es: ${NUMERO}"
echo "Texto: ${NOMBRE}_sufijo"   # llaves para delimitar

# Sustitución de comando (el resultado se convierte en el valor)
FECHA=$(date +%Y-%m-%d)
LINEAS=$(wc -l < archivo.txt)

# Variables de entorno (disponibles globalmente)
export MI_VAR="valor"            # disponible para subprocesos
```

**Variables especiales del sistema:**

| Variable | Significado |
|----------|-------------|
| `$HOME`  | Directorio personal del usuario |
| `$USER`  | Nombre del usuario |
| `$PATH`  | Directorios donde se buscan ejecutables |
| `$PWD`   | Directorio actual |
| `$?`     | Código de salida del último comando (0 = éxito) |
| `$$`     | PID del proceso actual |

In [ ]:
Path("variables.sh").write_text("""\
#!/usr/bin/env bash

# Declarar variables
PROYECTO="sistema_biblioteca"
VERSION="1.0.0"
FECHA=$(date +%Y-%m-%d)

# Usar variables
echo "Proyecto: $PROYECTO"
echo "Versión:  $VERSION"
echo "Fecha:    $FECHA"

# Variables del sistema
echo "Usuario:  $USER"
echo "Home:     $HOME"
echo "Pwd:      $PWD"

# Operaciones aritméticas (con dobles paréntesis)
A=10
B=3
echo "$A + $B = $((A + B))"
echo "$A * $B = $((A * B))"
echo "$A / $B = $((A / B))"

# Código de salida
ls /tmp > /dev/null 2>&1
echo "ls /tmp funcionó: $?"

ls /ruta_inexistente > /dev/null 2>&1
echo "ls ruta_inexistente falló: $?"
""", encoding="utf-8")

!bash variables.sh

---
## 3. Argumentos

Los argumentos que pasas al script al ejecutarlo están disponibles como variables posicionales:

```bash
bash mi_script.sh arg1 arg2 arg3

# Dentro del script:
$0    # nombre del script
$1    # primer argumento
$2    # segundo argumento
$@    # todos los argumentos (como lista)
$*    # todos los argumentos (como string)
$#    # número de argumentos
```

In [ ]:
Path("con_argumentos.sh").write_text("""\
#!/usr/bin/env bash
# Uso: bash con_argumentos.sh <nombre> <version>

NOMBRE=${1:-"proyecto"}     # valor por defecto si no se pasa
VERSION=${2:-"0.0.1"}

echo "Script: $0"
echo "Nombre: $NOMBRE"
echo "Versión: $VERSION"
echo "Argumentos recibidos: $#"
echo "Todos: $@"

# Iterar sobre todos los argumentos
echo "\nRecorriendo argumentos:"
for arg in "$@"; do
    echo "  - $arg"
done
""", encoding="utf-8")

print("=== Sin argumentos (valores por defecto) ===")
!bash con_argumentos.sh

print("\n=== Con argumentos ===")
!bash con_argumentos.sh mi_proyecto 2.1.0 extra

---
## 4. Condicionales

```bash
if [ condición ]; then
    # comandos si verdadero
elif [ otra_condición ]; then
    # comandos si elif verdadero
else
    # comandos si todo falso
fi
```

**Comparación de strings:**

| Operador | Significa |
|----------|-----------|
| `"$a" = "$b"` | igual |
| `"$a" != "$b"` | distinto |
| `-z "$a"` | cadena vacía |
| `-n "$a"` | cadena no vacía |

**Comparación numérica:**

| Operador | Significa |
|----------|-----------|
| `-eq` | igual |
| `-ne` | distinto |
| `-lt` | menor que |
| `-le` | menor o igual |
| `-gt` | mayor que |
| `-ge` | mayor o igual |

**Archivos y directorios:**

| Operador | Significa |
|----------|-----------|
| `-f archivo` | existe y es un archivo |
| `-d dir` | existe y es un directorio |
| `-e ruta` | existe (cualquier tipo) |
| `-r archivo` | tiene permisos de lectura |
| `-x archivo` | es ejecutable |

In [ ]:
Path("condicionales.sh").write_text("""\
#!/usr/bin/env bash

ARCHIVO=$1

# Verificar argumento
if [ -z "$ARCHIVO" ]; then
    echo "Error: debes pasar un archivo como argumento"
    exit 1
fi

# Comprobar si existe
if [ -e "$ARCHIVO" ]; then
    if [ -f "$ARCHIVO" ]; then
        LINEAS=$(wc -l < "$ARCHIVO")
        echo "Archivo encontrado: $ARCHIVO"
        echo "Líneas: $LINEAS"

        if [ $LINEAS -gt 10 ]; then
            echo "Es un archivo grande (más de 10 líneas)"
        else
            echo "Es un archivo pequeño (10 líneas o menos)"
        fi

    elif [ -d "$ARCHIVO" ]; then
        echo "Es un directorio, no un archivo"
    fi
else
    echo "No existe: $ARCHIVO"
    exit 1
fi
""", encoding="utf-8")

print("=== Sin argumento ===")
!bash condicionales.sh

print("\n=== Archivo existente ===")
!bash condicionales.sh variables.sh

print("\n=== Archivo inexistente ===")
!bash condicionales.sh no_existe.txt

---
## 5. Bucles

### `for` — iterar sobre una lista

```bash
# Iterar sobre lista literal
for item in a b c d; do
    echo "$item"
done

# Iterar sobre archivos (glob)
for archivo in *.py; do
    echo "Procesando: $archivo"
done

# Iterar sobre rango numérico
for i in $(seq 1 10); do
    echo "Iteración $i"
done

# Estilo C
for ((i=0; i<5; i++)); do
    echo $i
done
```

### `while` — iterar mientras condición sea verdadera

```bash
CONTADOR=0
while [ $CONTADOR -lt 5 ]; do
    echo "Contador: $CONTADOR"
    ((CONTADOR++))
done

# Leer archivo línea a línea
while IFS= read -r linea; do
    echo "Línea: $linea"
done < archivo.txt
```

In [ ]:
Path("bucles.sh").write_text("""\
#!/usr/bin/env bash

echo "=== for sobre lista ==="
for extension in py sh txt csv log; do
    COUNT=$(find . -name "*.$extension" | wc -l)
    echo "  .$extension: $COUNT archivos"
done

echo ""
echo "=== for sobre glob ==="
for script in *.sh; do
    LINEAS=$(wc -l < "$script")
    echo "  $script: $LINEAS líneas"
done

echo ""
echo "=== for numérico (simulación de 5 épocas) ==="
for epoch in $(seq 1 5); do
    echo "  Epoch $epoch/5 — procesando..."
done

echo ""
echo "=== while leyendo archivo ==="
CONTADOR=0
while IFS= read -r linea; do
    ((CONTADOR++))
done < variables.sh
echo "  variables.sh tiene $CONTADOR líneas"
""", encoding="utf-8")

!bash bucles.sh

---
## 6. Funciones

```bash
# Declarar una función
nombre_funcion() {
    local variable_local="valor"   # local: solo visible en la función
    echo "$variable_local"
    echo "Argumento 1: $1"
    return 0  # código de salida (0 = éxito)
}

# Llamar a la función
nombre_funcion "argumento"

# Capturar la salida de una función
resultado=$(nombre_funcion "argumento")
echo "Resultado: $resultado"
```

**Puntos importantes:**
- Hay que declarar las funciones **antes** de usarlas.
- `local` evita contaminar el scope global.
- `return` devuelve un código de salida (0-255), no un valor. Para devolver valores, usa `echo` y captura con `$(...)`.

In [ ]:
Path("funciones.sh").write_text("""\
#!/usr/bin/env bash

# Función: imprimir separador con título
titulo() {
    local texto="$1"
    local largo=${#texto}
    printf '=%.0s' $(seq 1 $((largo + 4)))
    echo ""
    echo "  $texto"
    printf '=%.0s' $(seq 1 $((largo + 4)))
    echo ""
}

# Función: verificar si un directorio existe, crearlo si no
asegurar_directorio() {
    local dir="$1"
    if [ ! -d "$dir" ]; then
        mkdir -p "$dir"
        echo "  Creado: $dir"
    else
        echo "  Ya existe: $dir"
    fi
}

# Función: contar archivos por extensión
contar_archivos() {
    local directorio="$1"
    local extension="$2"
    local count
    count=$(find "$directorio" -name "*.$extension" -type f 2>/dev/null | wc -l)
    echo $count
}

# --- Programa principal ---
titulo "Demo de Funciones"

echo ""
titulo "Verificar Directorios"
asegurar_directorio "/tmp/test_bash_dir"
asegurar_directorio "/tmp/test_bash_dir"

echo ""
titulo "Contar Scripts"
N=$(contar_archivos "." "sh")
echo "  Scripts .sh en el directorio actual: $N"
""", encoding="utf-8")

!bash funciones.sh

---
## 7. Práctica — Scripts de automatización para proyectos IA

---

### Ejercicio 1 — Script de verificación de entorno

**Enunciado:** Escribe `verificar_entorno.sh` que comprueba que el entorno de desarrollo está correctamente configurado:

1. Verifica que Python 3 está instalado y muestra su versión.
2. Verifica que Git está instalado.
3. Verifica que el directorio `~/ml_proyecto` existe.
4. Verifica que existe el archivo `requirements.txt` dentro de él.
5. Muestra un resumen: cuántas verificaciones pasaron y cuántas fallaron.
6. Sale con código 1 si alguna verificación falló.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
from pathlib import Path


<details>
<summary>💡 Ver solución — Ejercicio 1</summary>

```bash
#!/usr/bin/env bash
# verificar_entorno.sh — comprueba el entorno de desarrollo

PASARON=0
FALLARON=0

check() {
    local descripcion="$1"
    local resultado="$2"   # 0 = ok, 1 = fallo
    local detalle="$3"

    if [ "$resultado" -eq 0 ]; then
        echo "  ✅ $descripcion${detalle:+ — $detalle}"
        ((PASARON++))
    else
        echo "  ❌ $descripcion"
        ((FALLARON++))
    fi
}

echo "============================="
echo "  VERIFICACIÓN DE ENTORNO"
echo "============================="
echo ""

# 1. Python 3
if command -v python3 &> /dev/null; then
    VERSION=$(python3 --version 2>&1)
    check "Python 3 instalado" 0 "$VERSION"
else
    check "Python 3 instalado" 1
fi

# 2. Git
if command -v git &> /dev/null; then
    GIT_V=$(git --version)
    check "Git instalado" 0 "$GIT_V"
else
    check "Git instalado" 1
fi

# 3. Directorio ml_proyecto
if [ -d "$HOME/ml_proyecto" ]; then
    check "Directorio ~/ml_proyecto" 0
else
    check "Directorio ~/ml_proyecto" 1
fi

# 4. requirements.txt
if [ -f "$HOME/ml_proyecto/requirements.txt" ]; then
    check "requirements.txt existe" 0
else
    check "requirements.txt existe" 1
fi

# Resumen
echo ""
echo "Verificaciones: $PASARON pasaron, $FALLARON fallaron"

if [ $FALLARON -gt 0 ]; then
    echo "⚠️  Hay verificaciones fallidas"
    exit 1
else
    echo "✅ Entorno OK"
    exit 0
fi
```

**`command -v nombre`** es la forma correcta de verificar si un ejecutable existe en el PATH (más portable que `which`).
</details>

---
### Ejercicio 2 — Script de lanzamiento de experimentos

**Enunciado:** Escribe `lanzar_experimento.sh` que simula lanzar varios experimentos de ML:

```bash
bash lanzar_experimento.sh <nombre_proyecto> <lr1> <lr2> <lr3>
# Ejemplo: bash lanzar_experimento.sh mi_modelo 0.001 0.01 0.1
```

El script debe:
1. Verificar que se pasaron al menos dos argumentos.
2. Crear la carpeta `experimentos/<nombre_proyecto>/<fecha>/`.
3. Para cada learning rate, crear un archivo `config_lr_<lr>.yaml` con el contenido: `learning_rate: <lr>` y un timestamp.
4. Simular el entrenamiento: para cada config, imprimir `Lanzando experimento con lr=<lr>...` y crear un log vacío.
5. Al final, listar todos los archivos creados.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 2</summary>

```bash
#!/usr/bin/env bash
# lanzar_experimento.sh — simula lanzar experimentos con distintos LR

# 1. Verificar argumentos
if [ $# -lt 2 ]; then
    echo "Uso: $0 <nombre_proyecto> <lr1> [lr2] [lr3...]"
    echo "Ejemplo: $0 red_v1 0.001 0.01 0.1"
    exit 1
fi

PROYECTO=$1
shift             # elimina $1, los LR quedan en $@
FECHA=$(date +%Y%m%d_%H%M%S)
BASE="experimentos/$PROYECTO/$FECHA"

echo "======================================"
echo "  EXPERIMENTO: $PROYECTO"
echo "  Fecha: $FECHA"
echo "  Learning rates: $@"
echo "======================================"

# 2. Crear directorio base
mkdir -p "$BASE"
echo "Directorio creado: $BASE"

# 3 y 4. Para cada learning rate
for lr in "$@"; do
    CONFIG="$BASE/config_lr_${lr}.yaml"
    LOG="$BASE/train_lr_${lr}.log"

    # Crear config
    cat > "$CONFIG" << EOF
# Config generado automáticamente
proyecto: $PROYECTO
fecha: $FECHA
learning_rate: $lr
epochs: 50
batch_size: 32
EOF

    echo "Lanzando experimento con lr=$lr..."
    touch "$LOG"    # en un caso real: python train.py --config $CONFIG &> $LOG
    echo "  Config: $CONFIG"
    echo "  Log:    $LOG"
done

# 5. Listar archivos creados
echo ""
echo "Archivos creados:"
find "$BASE" -type f
echo ""
echo "✅ $(find "$BASE" -type f | wc -l) archivos generados"
```

**`shift`** desplaza los argumentos: después de `shift`, `$1` pasa a ser el anterior `$2`, y los LRs que quedan en `$@` se pueden recorrer con `for`.

**`cat << EOF ... EOF`** es un *heredoc* — forma cómoda de escribir texto multilínea en un archivo.
</details>

---
### Ejercicio 3 — Script de limpieza de proyecto

**Enunciado:** Escribe `limpiar_proyecto.sh <directorio>` que elimina los archivos temporales y cachés de un proyecto Python:

1. Verifica que el directorio existe y es accesible.
2. Busca y elimina: `__pycache__/`, `*.pyc`, `*.pyo`, `*.tmp`, `*.log` (con `-delete`).
3. Informa cuántos archivos/carpetas borró de cada tipo.
4. Muestra el espacio total recuperado antes y después (usa `du -sh`).
5. Pide confirmación antes de borrar (muestra qué va a borrar y espera `s/n`).

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


---
### Ejercicio 4 — Script de monitoreo de logs

**Enunciado:** Escribe `monitorear_logs.sh <directorio_logs>` que genera un informe del estado de los logs:

1. Lista todos los `.log` ordenados por fecha de modificación (más reciente primero).
2. Para cada archivo, muestra: nombre, tamaño, número de líneas, número de errores.
3. Calcula el total de errores en todos los logs.
4. Guarda el informe en `resumen_logs_<fecha>.txt`.
5. Si hay más de 10 errores en total, añade una alerta al final del informe.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 4</summary>

```bash
#!/usr/bin/env bash
# monitorear_logs.sh — informe de logs

DIR=${1:-"."}
FECHA=$(date +%Y%m%d_%H%M%S)
INFORME="resumen_logs_${FECHA}.txt"

if [ ! -d "$DIR" ]; then
    echo "Error: '$DIR' no es un directorio" >&2
    exit 1
fi

TOTAL_ERRORES=0

# Cabecera
{
echo "INFORME DE LOGS — $(date)"
echo "Directorio: $DIR"
echo ""
printf '%-30s %8s %8s %8s\n' "ARCHIVO" "TAMAÑO" "LÍNEAS" "ERRORES"
printf '%s\n' "$(printf '─%.0s' {1..58})"

# 1 y 2. Logs ordenados por fecha
while IFS= read -r log; do
    nombre=$(basename "$log")
    tamano=$(du -sh "$log" | cut -f1)
    lineas=$(wc -l < "$log")
    errores=$(grep -c 'ERROR' "$log" 2>/dev/null || echo 0)

    printf '%-30s %8s %8s %8s\n' "$nombre" "$tamano" "$lineas" "$errores"
    TOTAL_ERRORES=$((TOTAL_ERRORES + errores))
done < <(find "$DIR" -name "*.log" -type f -printf '%T@ %p\n' | sort -rn | cut -d' ' -f2-)

# 3. Total
echo ""
echo "Total de errores: $TOTAL_ERRORES"

# 5. Alerta
if [ $TOTAL_ERRORES -gt 10 ]; then
    echo ""
    echo "⚠️  ALERTA: Se encontraron $TOTAL_ERRORES errores (umbral: 10)"
    echo "   Revisa los archivos de log para identificar el problema."
fi

} | tee "$INFORME"

echo ""
echo "Informe guardado en: $INFORME"
```

**`{ ... } | tee archivo`** envía el bloque de comandos tanto a la pantalla como al archivo simultáneamente. **`< <(...)`** es *process substitution* — una forma avanzada de alimentar un `while` con la salida de otro comando.
</details>

---
### Ejercicio 5 (reto) — Script de backup con rotación

**Enunciado:** Escribe `backup_rotacion.sh <origen> <destino> <max_backups>` que implementa un sistema de backup con rotación automática:

1. Crea un backup del directorio `origen` en `destino/backup_YYYYMMDD_HHMMSS.tar.gz`.
2. Después de crear el backup, cuenta cuántos hay en `destino/`.
3. Si hay más de `max_backups`, elimina los más antiguos (mantiene solo los `max_backups` más recientes).
4. Verifica que el backup se creó correctamente (que el archivo existe y tiene tamaño > 0).
5. Genera un log de la operación en `destino/backup.log` con: fecha, tamaño del backup, backups eliminados.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 5</summary>

```bash
#!/usr/bin/env bash
set -euo pipefail
# backup_rotacion.sh — backup con rotación automática

ORIGEN=${1:?"Argumento 1 requerido: directorio origen"}
DESTINO=${2:?"Argumento 2 requerido: directorio destino"}
MAX=${3:-5}  # por defecto, conservar 5 backups

FECHA=$(date +%Y%m%d_%H%M%S)
NOMBRE_BACKUP="backup_${FECHA}.tar.gz"
RUTA_BACKUP="$DESTINO/$NOMBRE_BACKUP"
LOG="$DESTINO/backup.log"

mkdir -p "$DESTINO"

log_msg() {
    echo "[$(date '+%Y-%m-%d %H:%M:%S')] $*" | tee -a "$LOG"
}

log_msg "=== Inicio backup ==="
log_msg "Origen:  $ORIGEN"
log_msg "Destino: $RUTA_BACKUP"

# 1. Crear backup
if tar -czf "$RUTA_BACKUP" -C "$(dirname "$ORIGEN")" "$(basename "$ORIGEN")" 2>/dev/null; then
    TAMANO=$(du -sh "$RUTA_BACKUP" | cut -f1)
    log_msg "Backup creado: $TAMANO"
else
    log_msg "ERROR: fallo al crear el backup"
    exit 1
fi

# 4. Verificar que existe y tiene tamaño
if [ ! -s "$RUTA_BACKUP" ]; then
    log_msg "ERROR: el backup está vacío"
    exit 1
fi
log_msg "Verificación OK"

# 2 y 3. Rotación: eliminar backups en exceso
TOTAL=$(find "$DESTINO" -maxdepth 1 -name "backup_*.tar.gz" | wc -l)
log_msg "Backups existentes: $TOTAL (máximo: $MAX)"

ELIMINADOS=0
while [ "$TOTAL" -gt "$MAX" ]; do
    MAS_ANTIGUO=$(find "$DESTINO" -maxdepth 1 -name "backup_*.tar.gz" | sort | head -1)
    log_msg "Eliminando: $(basename "$MAS_ANTIGUO")"
    rm "$MAS_ANTIGUO"
    ((TOTAL--))
    ((ELIMINADOS++))
done

log_msg "Backups eliminados: $ELIMINADOS"
log_msg "Backups actuales: $(find "$DESTINO" -maxdepth 1 -name 'backup_*.tar.gz' | wc -l)"
log_msg "=== Fin backup OK ==="
echo ""
echo "✅ Backup completado: $RUTA_BACKUP ($TAMANO)"
```

**`${VAR:?"mensaje"}`** falla con el mensaje indicado si la variable no está definida — mejor que verificar manualmente con `if`.

**`-s`** en los condicionales de archivos comprueba que el archivo existe Y tiene tamaño mayor que cero.
</details>

---
## Resumen de la clase

```bash
# ─── Estructura ────────────────────────────────────────────
#!/usr/bin/env bash          # shebang
set -euo pipefail            # salir en error, vars no definidas
chmod +x script.sh           # hacer ejecutable
bash script.sh               # ejecutar

# ─── Variables ─────────────────────────────────────────────
VAR="valor"                  # declarar (sin espacios)
echo "$VAR"                  # usar
RESULTADO=$(comando)         # sustitución de comando
$((A + B))                   # aritmética
${VAR:-"default"}           # valor por defecto

# ─── Argumentos ────────────────────────────────────────────
$1 $2 ...                    # argumentos posicionales
$@ / $*                      # todos los argumentos
$#                           # número de argumentos
shift                        # desplazar argumentos

# ─── Condicionales ─────────────────────────────────────────
if [ cond ]; then ... fi     # básico
[ -f archivo ]               # es archivo
[ -d dir ]                   # es directorio
[ -z "$VAR" ]                # vacía
[ $A -gt $B ]                # mayor que
command -v prog              # está instalado

# ─── Bucles ────────────────────────────────────────────────
for item in lista; do ... done
for f in *.py; do ... done   # sobre archivos
for i in $(seq 1 10); do ... done
while [ cond ]; do ... done
while IFS= read -r line; do ... done < archivo

# ─── Funciones ─────────────────────────────────────────────
nombre() { local v="..."; echo "$v"; }
resultado=$(nombre)
```

## ¿Qué viene en la próxima clase?

La **Clase 5** es el proyecto integrador: escribimos un script completo de automatización que combina todo lo del Módulo 3 — gestión de archivos, búsqueda, redirecciones y Bash scripting.